# Sentinel-1 Flood Frequency (Google Earth Engine)

Builds a flood-frequency layer at 30 m from Sentinel-1 VH radar: for each target date it compares the image around the event with a pre-event baseline and flags pixels that darken by more than 3 dB (change detection), masks steep slopes and permanent water, and counts the detections across dates.

Requires a Google Earth Engine account. Replace `YOUR_GEE_PROJECT_ID` with your own project. The classified raster is produced by the SAR classification notebook.

## 0. Authentication

In [ ]:
# Google Earth Engine Python API
import ee

# Authenticate with Earth Engine through a local browser flow
ee.Authenticate(auth_mode="localhost")
# Initialize Earth Engine with your own Google Cloud project
ee.Initialize(project="YOUR_GEE_PROJECT_ID")

## 1. Adjustable parameters

In [ ]:
# Buffer around each historical flood point, in metres
BUFFER_AOI_M = 5000
# Change-detection threshold: a drop of more than 3 dB in VH backscatter is flagged as flooding
THRESHOLD_DB = -3.0
# Maximum terrain slope in degrees; steeper pixels are excluded
SLOPE_MAX_DEG = 5
# Maximum JRC water occurrence; pixels with permanent water are excluded
JRC_WATER_OCCURRENCE_MAX = 90
# Minimum size of connected flooded patches at the original resolution
CONNECTED_PIXEL_MIN = 8
# Export resolution in metres
EXPORT_SCALE_M = 30
# Length of the pre-event baseline window, in days
BASELINE_LOOKBACK_DAYS = 36
# Gap between the baseline and the event window, in days
BASELINE_GAP_DAYS = 12
# Half-width of the event window around each target date, in days
EVENT_WINDOW_DAYS = 6

# Google Drive folder that receives the export
DRIVE_FOLDER = "EFCI_Ecuador_Sentinel1"

# Minimum size of connected flooded patches at 30 m
CONNECTED_PIXEL_MIN_30M = 2
# Prefix of the exported file
FILE_PREFIX_30M = "efci_flood_frequency_2015_2025_30m"

## 2. Target dates

In [ ]:
# Target dates grouped by El Nino episode
FECHAS_OBJETIVO = {
    "2015": ["2015-03-20", "2015-04-26", "2015-02-07"],
    "2016": ["2016-01-25", "2016-03-06", "2016-03-03"],
    "2017": ["2017-04-08", "2017-03-24", "2017-02-08"],
    "2023": ["2023-04-18", "2023-03-09", "2023-03-23"],
    "2024": ["2024-02-21", "2024-02-20", "2024-02-19"],
}
# Flat list of all target dates
TODAS_LAS_FECHAS = [f for lista in FECHAS_OBJETIVO.values() for f in lista]
# Check there are 15 dates (3 per episode, 5 episodes)
assert len(TODAS_LAS_FECHAS) == 15, "se esperaban 15 fechas objetivo (3 x 5 episodios)"
# Print the number of target dates
print(f"{len(TODAS_LAS_FECHAS)} fechas objetivo cargadas.")

## 3. Area of interest: historical flood event points

In [ ]:
# JSON reader
import json
# Object-oriented file paths
from pathlib import Path

# Try the standard quantile cut
try:
    # Folder of the notebook when run as a script
    base_dir = Path(__file__).parent
except NameError:
    # Folder of the notebook when run interactively
    base_dir = Path.cwd()

# Path to the file with the historical flood points
aoi_path = base_dir / "aoi_flood_points.json"
# Check the points file exists
assert aoi_path.exists(), (
    f"No se encontró {aoi_path}. Sube 'aoi_flood_points.json' a la misma "
    "carpeta que este notebook (en Colab: panel de archivos a la izquierda, "
    "arrastrar el archivo a /content/)."
)

# Open the points file
with aoi_path.open("r", encoding="utf-8") as f:
    # Read the points
    aoi_data = json.load(f)

# List of (longitude, latitude) pairs
puntos = aoi_data["puntos_lonlat"]
# Print the number of points
print(f"Puntos históricos de eventos 'flood' cargados: {len(puntos)}")

# Earth Engine multipoint geometry
geom_puntos = ee.Geometry.MultiPoint(puntos)
# Bounding box of the buffered points
AOI = geom_puntos.buffer(BUFFER_AOI_M).bounds()
# Union of the buffers around the points
AOI_UNION = geom_puntos.buffer(BUFFER_AOI_M)

## 4. Ancillary layers: slope (DEM) and permanent water (JRC)

In [ ]:
# Digital elevation model (SRTM)
dem = ee.Image("USGS/SRTMGL1_003")
# Slope in degrees
slope = ee.Terrain.slope(dem)
# Mask of pixels with gentle slopes
mascara_pendiente = slope.lte(SLOPE_MAX_DEG)

# JRC Global Surface Water dataset
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
# Mask excluding permanent water
mascara_agua_permanente = jrc.select("occurrence").lt(JRC_WATER_OCCURRENCE_MAX).unmask(1)

# Combined mask clipped to the area of interest
mascara_base = mascara_pendiente.And(mascara_agua_permanente).clip(AOI_UNION)
# Print that the masks are ready
print("Máscaras de pendiente y agua permanente listas.")

## 5. Per-date flood mask function

In [ ]:
# Sentinel-1 VH collection between two dates
def coleccion_s1_vh(fecha_ini, fecha_fin):
    """Sentinel-1 VH collection for a date range, filtered to the area of interest.
    
    Args:
        fecha_ini: Start date.
        fecha_fin: End date.
    
    Returns:
        Earth Engine image collection.
    """
    # Weighted sum of the four components
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(AOI_UNION)
        .filterDate(fecha_ini, fecha_fin)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .select("VH")
    )


# Compute the event and baseline windows for a target date
def ventanas_fecha(fecha_str):
    """Compute the event window and the baseline window for a target date.
    
    Args:
        fecha_str: Target date (YYYY-MM-DD).
    
    Returns:
        Event start, event end, baseline start and baseline end.
    """
    # Target date as an Earth Engine date
    fecha = ee.Date(fecha_str)
    # Start of the event window
    evento_ini = fecha.advance(-EVENT_WINDOW_DAYS, "day")
    # End of the event window
    evento_fin = fecha.advance(EVENT_WINDOW_DAYS, "day")
    # End of the baseline window
    ref_fin = fecha.advance(-BASELINE_GAP_DAYS, "day")
    # Start of the baseline window
    ref_ini = ref_fin.advance(-BASELINE_LOOKBACK_DAYS, "day")
    # Return the four dates
    return evento_ini, evento_fin, ref_ini, ref_fin


# Binary flood mask for one target date
def mascara_inundacion_fecha(fecha_str):
    """Build the binary flood mask for one target date by change detection.
    
    Args:
        fecha_str: Target date (YYYY-MM-DD).
    
    Returns:
        Image with 1 for flooded pixels and 0 elsewhere.
    """
    # Event and baseline windows
    evento_ini, evento_fin, ref_ini, ref_fin = ventanas_fecha(fecha_str)

    # Images around the event
    col_evento = coleccion_s1_vh(evento_ini, evento_fin)
    # Images in the baseline period
    col_referencia = coleccion_s1_vh(ref_ini, ref_fin)

    # Median image of the event period
    img_evento = ee.Image(ee.Algorithms.If(
        col_evento.size().gt(0), col_evento.median(), ee.Image.constant(0).rename("VH")
    ))
    # Median image of the baseline period
    img_referencia = ee.Image(ee.Algorithms.If(
        col_referencia.size().gt(0), col_referencia.median(), ee.Image.constant(0).rename("VH")
    ))

    # Resample the event image to the export grid
    img_evento = img_evento.resample("bilinear").reproject(crs="EPSG:4326", scale=EXPORT_SCALE_M)
    # Resample the baseline image to the export grid
    img_referencia = img_referencia.resample("bilinear").reproject(crs="EPSG:4326", scale=EXPORT_SCALE_M)

    # Difference in backscatter (dB) between event and baseline
    diferencia = img_evento.subtract(img_referencia).rename("dif_db")

    # Flag pixels that darkened beyond the threshold
    inundado = diferencia.lt(THRESHOLD_DB)
    # Apply the slope and permanent-water mask
    inundado = inundado.updateMask(mascara_base)

    # Check both collections contain images
    hay_datos = col_evento.size().gt(0).And(col_referencia.size().gt(0))
    # Mask everything when data is missing
    inundado = inundado.updateMask(ee.Image.constant(hay_datos))

    # Count connected flooded pixels
    conteo_conectado = inundado.connectedPixelCount(25, True)
    # Remove small isolated patches
    inundado_limpio = inundado.updateMask(conteo_conectado.gte(CONNECTED_PIXEL_MIN_30M))

    # Return the final mask for the date
    return inundado_limpio.unmask(0).rename("flood").clip(AOI_UNION).set("fecha", fecha_str)


# Print that the functions are defined
print("Funciones de detección de cambio definidas.")

## 5.1 Image availability diagnostic

In [ ]:
# Print a header for the availability check
print("Verificando disponibilidad de Sentinel-1 por fecha objetivo (evento AOI: Ecuador, zonas historicamente inundables)...\n")

# Dictionary of availability per date
disponibilidad = {}
# Loop over the target dates
for f in TODAS_LAS_FECHAS:
    # Event and baseline windows
    evento_ini, evento_fin, ref_ini, ref_fin = ventanas_fecha(f)
    # Number of event images
    n_evento = coleccion_s1_vh(evento_ini, evento_fin).size().getInfo()
    # Number of baseline images
    n_referencia = coleccion_s1_vh(ref_ini, ref_fin).size().getInfo()
    # Date is usable only with images in both periods
    ok = n_evento > 0 and n_referencia > 0
    # Store the result
    disponibilidad[f] = ok
    # Status label
    estado = "OK" if ok else "SIN DATOS -> se excluye"
    # Print the image counts per date
    print(f"  {f}: {n_evento:2d} imagenes evento, {n_referencia:2d} imagenes referencia  [{estado}]")

# Dates with coverage
FECHAS_VALIDAS = [f for f, ok in disponibilidad.items() if ok]
# Dates without coverage
FECHAS_EXCLUIDAS = [f for f, ok in disponibilidad.items() if not ok]

# Print how many dates have coverage
print(f"\n{len(FECHAS_VALIDAS)} de {len(TODAS_LAS_FECHAS)} fechas objetivo tienen cobertura Sentinel-1 completa.")
# Report excluded dates, if any
if FECHAS_EXCLUIDAS:
    # Print the excluded dates
    print(f"Excluidas por falta de cobertura: {FECHAS_EXCLUIDAS}")
    # Note that the maximum frequency equals the number of valid dates
    print("(Documentar esto en el reporte final: el techo real de la capa de")
    # Print the maximum possible frequency
    print(f" frecuencia es {len(FECHAS_VALIDAS)}, no {len(TODAS_LAS_FECHAS)}, si alguna fecha quedo excluida.)")

## 6. Build the masks (dates with data only) and aggregate into a frequency layer

In [ ]:
# Check at least one date has coverage
assert len(FECHAS_VALIDAS) > 0, "ninguna fecha objetivo tiene cobertura Sentinel-1 -- revisar el AOI o las fechas antes de continuar"

# Flood mask for every valid date
mascaras = [mascara_inundacion_fecha(f) for f in FECHAS_VALIDAS]
# Collection of masks
coleccion_mascaras = ee.ImageCollection(mascaras)

# Frequency layer: number of dates each pixel was flagged as flooded
frecuencia = coleccion_mascaras.sum().toUint8().rename("frecuencia_inundacion")
# Frequency layer at the export grid
frecuencia_export = frecuencia.reproject(crs="EPSG:4326", scale=EXPORT_SCALE_M).toUint8()

# Print the number of dates behind the layer
print(f"Capa de frecuencia construida sobre {len(FECHAS_VALIDAS)} fechas validas "
      f"(de {len(TODAS_LAS_FECHAS)} objetivo): valores posibles 0 a {len(FECHAS_VALIDAS)}.")

## 7. Export to Google Drive

In [ ]:
# Timing utilities
import time

# Define the export task to Google Drive
tarea_frecuencia = ee.batch.Export.image.toDrive(
    image=frecuencia_export,
    description="EFCI_flood_frequency_2015_2025_30m",
    folder=DRIVE_FOLDER,
    fileNamePrefix=FILE_PREFIX_30M,
    region=AOI,
    scale=EXPORT_SCALE_M,
    crs="EPSG:4326",
    maxPixels=1e10,
)
# Start the export
tarea_frecuencia.start()

# Print the task name and id
print(f"Tarea de exportación iniciada: EFCI_flood_frequency_2015_2025_30m (id: {tarea_frecuencia.id})")
# Print where to track the task
print("Seguimiento manual (si prefieres no esperar aquí): https://code.earthengine.google.com/tasks\n")

# Polling interval in seconds
INTERVALO_SEGUNDOS = 30
# Wait while the task is running
while tarea_frecuencia.active():
    # Current task status
    estado = tarea_frecuencia.status()
    # Current time
    marca_tiempo = time.strftime("%H:%M:%S")
    # Print the status
    print(f"[{marca_tiempo}] Estado: {estado['state']}")
    # Wait before checking again
    time.sleep(INTERVALO_SEGUNDOS)

# Final task status
estado_final = tarea_frecuencia.status()
# Print the final status
print(f"\nTarea finalizada con estado: {estado_final['state']}")
# Check whether the task failed
if estado_final["state"] == "FAILED":
    # Print the Earth Engine error message
    print("Mensaje de error de GEE:", estado_final.get("error_message", "(sin detalle)"))
    # Print a hint about missing coverage
    print("Revisa el diagnostico de la Seccion 5.1 -- si el error menciona bandas o")
    # Print a second hint
    print("imagenes faltantes, probablemente falte cobertura Sentinel-1 en alguna fecha/zona del AOI.")
else:
    # Print where to download the GeoTIFF
    print("Descarga el GeoTIFF desde Google Drive, carpeta:", DRIVE_FOLDER)

## 8. (Optional) Export the individual masks